In [1]:
# ============================================================
# SETUP: INSTALL LIBRARIES AND CREATE KNOWLEDGE BASE
# ============================================================

!pip install -q langchain-text-splitters chromadb sentence-transformers rank_bm25 pandas

from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi
import pandas as pd

# 1. Prepare documents
documents = {
    "ai_overview.txt": """
Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science
focused on building systems that can perform tasks which normally
require human intelligence.

Machine Learning is a subset of AI where systems learn patterns
from data instead of following hardcoded rules.

Deep Learning takes Machine Learning further by using neural
networks with many layers to automatically discover complex patterns
in data.

Retrieval-Augmented Generation, or RAG, combines a search system
with a language model so answers are grounded in retrieved documents.
""",

    "embeddings_in_practice.txt": """
Embeddings and Vector Search in Practice

Modern search engines use embeddings rather than simple keyword
matching to find results that match the meaning of a query.

Vector databases such as ChromaDB store embeddings and allow fast
similarity search across thousands or millions of documents.
""",

    "product_catalog.txt": """
Product Catalog Notes

The wireless noise-cancelling headphones are listed under product
code SKU-4521 and are currently our best-selling audio accessory.

The standing desk converter is listed under product code SKU-7788
and ships within three business days.

Return requests for any product must reference the exact SKU code
so our warehouse team can locate the correct item quickly.
"""
}

# 2. Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=220,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# 3. Create ChromaDB client and embedding model
client = chromadb.Client()

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Safely recreate the collection if this cell is run again
try:
    client.delete_collection("rag_hybrid_demo")
except Exception:
    pass

collection = client.create_collection(
    name="rag_hybrid_demo",
    embedding_function=hf_embedder
)

# 4. Store chunks, metadata and IDs
all_chunks = []
all_chunk_ids = []
all_chunk_meta = []

for source, text in documents.items():
    doc_chunks = splitter.split_text(text)

    doc_ids = [
        f"{source}_chunk_{i}"
        for i in range(len(doc_chunks))
    ]

    doc_meta = [
        {"source": source, "chunk_index": i}
        for i in range(len(doc_chunks))
    ]

    collection.add(
        documents=doc_chunks,
        metadatas=doc_meta,
        ids=doc_ids
    )

    all_chunks.extend(doc_chunks)
    all_chunk_ids.extend(doc_ids)
    all_chunk_meta.extend(doc_meta)

print("Setup completed successfully!")
print("Total documents:", len(documents))
print("Total chunks stored:", collection.count())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Setup completed successfully!
Total documents: 3
Total chunks stored: 9


In [2]:
# ============================================================
# SEMANTIC SEARCH
# ============================================================

def semantic_search(query, n_results=3):
    results = collection.query(
        query_texts=[query],
        n_results=min(n_results, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    output = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        output.append({
            "text": doc,
            "source": meta["source"],
            "similarity": round(1 - distance, 4)
        })

    return output


results = semantic_search("SKU-4521", n_results=3)

for result in results:
    print("Similarity:", result["similarity"])
    print("Source:", result["source"])
    print("Text:", result["text"])
    print("-" * 60)

Similarity: 0.3853
Source: product_catalog.txt
Text: Return requests for any product must reference the exact SKU code
so our warehouse team can locate the correct item quickly.
------------------------------------------------------------
Similarity: 0.3851
Source: product_catalog.txt
Text: Product Catalog Notes

The wireless noise-cancelling headphones are listed under product
code SKU-4521 and are currently our best-selling audio accessory.
------------------------------------------------------------
Similarity: 0.2492
Source: product_catalog.txt
Text: The standing desk converter is listed under product code SKU-7788
and ships within three business days.
------------------------------------------------------------


BM25 keyword search

In [3]:
# ============================================================
# BM25 KEYWORD SEARCH
# ============================================================

def tokenize(text):
    return text.lower().replace("-", " ").split()


def rebuild_bm25():
    global tokenized_chunks, bm25

    tokenized_chunks = [
        tokenize(chunk) for chunk in all_chunks
    ]

    bm25 = BM25Okapi(tokenized_chunks)


rebuild_bm25()


def bm25_search(query, n_results=3):
    scores = bm25.get_scores(tokenize(query))

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:n_results]

    output = []

    for idx in ranked_indices:
        if scores[idx] > 0:
            output.append({
                "text": all_chunks[idx],
                "source": all_chunk_meta[idx]["source"],
                "bm25_score": round(float(scores[idx]), 4)
            })

    return output


results = bm25_search("SKU-4521", n_results=3)

for result in results:
    print("BM25 score:", result["bm25_score"])
    print("Source:", result["source"])
    print("Text:", result["text"])
    print("-" * 60)

BM25 score: 2.2625
Source: product_catalog.txt
Text: Product Catalog Notes

The wireless noise-cancelling headphones are listed under product
code SKU-4521 and are currently our best-selling audio accessory.
------------------------------------------------------------
BM25 score: 0.6785
Source: product_catalog.txt
Text: The standing desk converter is listed under product code SKU-7788
and ships within three business days.
------------------------------------------------------------
BM25 score: 0.6205
Source: product_catalog.txt
Text: Return requests for any product must reference the exact SKU code
so our warehouse team can locate the correct item quickly.
------------------------------------------------------------


In [4]:
results = bm25_search(
    "a computer system that learns from experience"
)

print("BM25 results for a meaning-based query:")

if not results:
    print("(No keyword matches found)")

for result in results:
    print(result["text"])

BM25 results for a meaning-based query:
Retrieval-Augmented Generation, or RAG, combines a search system
with a language model so answers are grounded in retrieved documents.
Machine Learning is a subset of AI where systems learn patterns
from data instead of following hardcoded rules.
Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science
focused on building systems that can perform tasks which normally
require human intelligence.


Hybrid search using Reciprocal Rank Fusion (RRF)

In [5]:
# ============================================================
# HYBRID SEARCH USING RRF
# ============================================================

def hybrid_search(query, n_results=3, k=60):
    semantic_results = semantic_search(query, n_results=10)
    keyword_results = bm25_search(query, n_results=10)

    rrf_scores = {}
    chunk_lookup = {}

    # Add semantic rankings
    for rank, result in enumerate(semantic_results):
        key = result["text"]

        rrf_scores[key] = (
            rrf_scores.get(key, 0) + 1 / (k + rank + 1)
        )

        chunk_lookup[key] = result

    # Add keyword rankings
    for rank, result in enumerate(keyword_results):
        key = result["text"]

        rrf_scores[key] = (
            rrf_scores.get(key, 0) + 1 / (k + rank + 1)
        )

        chunk_lookup.setdefault(key, result)

    ranked = sorted(
        rrf_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )[:n_results]

    output = []

    for text, score in ranked:
        result = chunk_lookup[text]

        output.append({
            "text": text,
            "source": result["source"],
            "rrf_score": round(score, 5)
        })

    return output


results = hybrid_search("SKU-4521 product details")

for result in results:
    print("RRF score:", result["rrf_score"])
    print("Source:", result["source"])
    print("Text:", result["text"])
    print("-" * 60)

RRF score: 0.03252
Source: product_catalog.txt
Text: Product Catalog Notes

The wireless noise-cancelling headphones are listed under product
code SKU-4521 and are currently our best-selling audio accessory.
------------------------------------------------------------
RRF score: 0.03227
Source: product_catalog.txt
Text: Return requests for any product must reference the exact SKU code
so our warehouse team can locate the correct item quickly.
------------------------------------------------------------
RRF score: 0.032
Source: product_catalog.txt
Text: The standing desk converter is listed under product code SKU-7788
and ships within three business days.
------------------------------------------------------------


In [ ]:
# ============================================================
# COMPARE SEMANTIC, BM25 AND HYBRID SEARCH
# ============================================================

test_query = "What is SKU-4521?"

semantic_top = semantic_search(test_query, n_results=1)
keyword_top = bm25_search(test_query, n_results=1)
hybrid_top = hybrid_search(test_query, n_results=1)

comparison = pd.DataFrame({
    "Method": [
        "Semantic only",
        "Keyword (BM25) only",
        "Hybrid (RRF)"
    ],
    "Top Result": [
        semantic_top[0]["text"][:100] + "..."
        if semantic_top else "(no result)",

        keyword_top[0]["text"][:100] + "..."
        if keyword_top else "(no result)",

        hybrid_top[0]["text"][:100] + "..."
        if hybrid_top else "(no result)"
    ]
})

display(comparison)

Handle empty retrievals using a similarity threshold


In [6]:
# ============================================================
# EMPTY RETRIEVAL: SIMILARITY THRESHOLD
# ============================================================

def distance_to_similarity(distance):
    return 1 - distance


def retrieve_with_threshold(
    query,
    n_results=5,
    similarity_threshold=0.3
):
    results = collection.query(
        query_texts=[query],
        n_results=min(n_results, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    filtered = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = distance_to_similarity(distance)

        if similarity >= similarity_threshold:
            filtered.append({
                "text": doc,
                "source": meta["source"],
                "similarity": round(similarity, 4)
            })

    return filtered


empty_test = retrieve_with_threshold(
    "What's the best recipe for chocolate cake?",
    similarity_threshold=0.3
)

print("Chunks passing the threshold:", len(empty_test))

for result in empty_test:
    print(result)

Chunks passing the threshold: 0


Create smart_retrieve() with a fallback chain

In [7]:
# ============================================================
# SMART RETRIEVAL: SEMANTIC -> HYBRID -> EMPTY
# ============================================================

KNOWLEDGE_BASE_TOPICS = sorted(
    {meta["source"] for meta in all_chunk_meta}
)


def smart_retrieve(
    query,
    n_results=4,
    similarity_threshold=0.3,
    rrf_k=60
):
    # Step 1: Semantic search with a threshold
    primary = retrieve_with_threshold(
        query,
        n_results=n_results,
        similarity_threshold=similarity_threshold
    )

    if primary:
        return {
            "status": "ok",
            "method": "semantic",
            "chunks": primary
        }

    # Step 2: Check whether BM25 found actual keyword matches.
    # This avoids accepting arbitrary nearest chunks for unrelated queries.
    keyword_results = bm25_search(query, n_results=10)

    if keyword_results:
        hybrid_results = hybrid_search(
            query,
            n_results=n_results,
            k=rrf_k
        )

        if hybrid_results:
            return {
                "status": "ok",
                "method": "hybrid_fallback",
                "chunks": hybrid_results
            }

    # Step 3: Return an honest answer when no retrieval works
    return {
        "status": "empty",
        "method": None,
        "chunks": [],
        "message": (
            "I don't know based on the provided documents. "
            "This knowledge base currently covers: "
            + ", ".join(KNOWLEDGE_BASE_TOPICS)
            + "."
        )
    }


# Test 1: In-scope question
result = smart_retrieve("What is deep learning?")

print("Test 1:", result["status"], "|", result["method"])


# Test 2: Exact product code
result = smart_retrieve("Tell me about SKU-4521")

print("Test 2:", result["status"], "|", result["method"])


# Test 3: Out-of-scope question
result = smart_retrieve(
    "What's the best recipe for chocolate cake?"
)

print("Test 3:", result["status"])
print(result.get("message", "Relevant context found."))

Test 1: ok | semantic
Test 2: ok | semantic
Test 3: ok
Relevant context found.


Format retrieved context for an LLM

In [8]:
# ============================================================
# SAFE RAG PROMPT
# ============================================================

def format_context_for_llm(query, chunks):
    context_text = "\n\n".join(
        f"[Source: {chunk['source']}]\n{chunk['text']}"
        for chunk in chunks
    )

    return (
        "Answer the question using only the context below. "
        "If the answer is not present, say you don't know.\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {query}\n\n"
        "Answer:"
    )


def safe_rag_prompt(query):
    retrieval = smart_retrieve(query)

    if retrieval["status"] == "empty":
        return retrieval["message"]

    prompt = format_context_for_llm(
        query,
        retrieval["chunks"]
    )

    return (
        f"[Retrieved using: {retrieval['method']}]\n\n"
        f"{prompt}"
    )


print(safe_rag_prompt("What is SKU-4521?"))

print("\n" + "=" * 70 + "\n")

print(
    safe_rag_prompt(
        "What's the best recipe for chocolate cake?"
    )
)

[Retrieved using: semantic]

Answer the question using only the context below. If the answer is not present, say you don't know.

Context:
[Source: product_catalog.txt]
Product Catalog Notes

The wireless noise-cancelling headphones are listed under product
code SKU-4521 and are currently our best-selling audio accessory.

[Source: product_catalog.txt]
Return requests for any product must reference the exact SKU code
so our warehouse team can locate the correct item quickly.

Question: What is SKU-4521?

Answer:


[Retrieved using: hybrid_fallback]

Answer the question using only the context below. If the answer is not present, say you don't know.

Context:
[Source: product_catalog.txt]
Return requests for any product must reference the exact SKU code
so our warehouse team can locate the correct item quickly.

[Source: ai_overview.txt]
Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science
focused on building systems that can perform task

Exercise 1 — Change the RRF constant from 60 to 10

In [ ]:
# ============================================================
# EXERCISE 1: COMPARE RRF k=60 AND k=10
# ============================================================

query = "What is SKU-4521?"

results_k60 = hybrid_search(query, n_results=3, k=60)
results_k10 = hybrid_search(query, n_results=3, k=10)


print("RESULTS WITH k = 60")
for i, result in enumerate(results_k60, start=1):
    print(
        i,
        result["rrf_score"],
        result["source"],
        result["text"][:100]
    )


print("\nRESULTS WITH k = 10")
for i, result in enumerate(results_k10, start=1):
    print(
        i,
        result["rrf_score"],
        result["source"],
        result["text"][:100]
    )

Exercise 2 — Add a fourth document with a new product code

In [9]:
# ============================================================
# EXERCISE 2: ADD A FOURTH DOCUMENT
# ============================================================

fourth_document = """
Product Catalog Update

The ergonomic office chair is listed under product code SKU-9912.
It includes adjustable armrests and lumbar support.

The chair is available for office and home use.
Orders are processed within two business days.
"""

source_name = "new_products.txt"

# Split into chunks
fourth_chunks = splitter.split_text(fourth_document)

# Create unique IDs and metadata
fourth_ids = [
    f"{source_name}_chunk_{i}"
    for i in range(len(fourth_chunks))
]

fourth_metadata = [
    {"source": source_name, "chunk_index": i}
    for i in range(len(fourth_chunks))
]

# Store in ChromaDB
collection.add(
    documents=fourth_chunks,
    metadatas=fourth_metadata,
    ids=fourth_ids
)

# Update Python's parallel lists used by BM25
all_chunks.extend(fourth_chunks)
all_chunk_ids.extend(fourth_ids)
all_chunk_meta.extend(fourth_metadata)

# Rebuild BM25 to include the new document
rebuild_bm25()

print("Fourth document added!")
print("Total chunks:", collection.count())


# Search for the new product code
result = smart_retrieve("Tell me about SKU-9912")

print("\nRetrieval status:", result["status"])
print("Method:", result["method"])

for chunk in result["chunks"]:
    print("\nSource:", chunk["source"])
    print(chunk["text"])

Fourth document added!
Total chunks: 11

Retrieval status: ok
Method: semantic

Source: product_catalog.txt
Return requests for any product must reference the exact SKU code
so our warehouse team can locate the correct item quickly.

Source: product_catalog.txt
Product Catalog Notes

The wireless noise-cancelling headphones are listed under product
code SKU-4521 and are currently our best-selling audio accessory.


Exercise 3 — Add a stopword-removal fallback

In [10]:
# ============================================================
# EXERCISE 3: STOPWORD-REMOVAL FALLBACK
# ============================================================

STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were",
    "what", "which", "who", "how", "why",
    "for", "to", "of", "in", "on", "and",
    "does", "do", "with", "me", "tell", "about"
}


def remove_stopwords(query):
    words = tokenize(query)
    remaining_words = [
        word for word in words
        if word not in STOPWORDS
    ]
    return " ".join(remaining_words)


def smart_retrieve_with_stopwords(
    query,
    n_results=4,
    similarity_threshold=0.3,
    rrf_k=60
):
    # First, use the normal retrieval pipeline
    result = smart_retrieve(
        query,
        n_results=n_results,
        similarity_threshold=similarity_threshold,
        rrf_k=rrf_k
    )

    if result["status"] == "ok":
        return result

    # Fourth step: remove stopwords and retry keyword retrieval
    cleaned_query = remove_stopwords(query)

    if not cleaned_query:
        return result

    cleaned_keywords = bm25_search(
        cleaned_query,
        n_results=n_results
    )

    if cleaned_keywords:
        cleaned_hybrid = hybrid_search(
            cleaned_query,
            n_results=n_results,
            k=rrf_k
        )

        if cleaned_hybrid:
            return {
                "status": "ok",
                "method": "stopword_fallback",
                "chunks": cleaned_hybrid
            }

    return result


# Test the new fallback
query = "Can you tell me about the SKU-9912 product?"

result = smart_retrieve_with_stopwords(query)

print("Original query:", query)
print("Cleaned query:", remove_stopwords(query))
print("Status:", result["status"])
print("Method:", result["method"])

if result["status"] == "empty":
    print(result["message"])
else:
    for chunk in result["chunks"]:
        print("\nSource:", chunk["source"])
        print(chunk["text"])

Original query: Can you tell me about the SKU-9912 product?
Cleaned query: can you sku 9912 product?
Status: ok
Method: semantic

Source: product_catalog.txt
Return requests for any product must reference the exact SKU code
so our warehouse team can locate the correct item quickly.

Source: product_catalog.txt
Product Catalog Notes

The wireless noise-cancelling headphones are listed under product
code SKU-4521 and are currently our best-selling audio accessory.

Source: new_products.txt
Product Catalog Update

The ergonomic office chair is listed under product code SKU-9912.
It includes adjustable armrests and lumbar support.
